# Playbook RAG — building and measuring the retrieval layer

The SOC agent's first move on a FortiSOAR alert is to find the playbook that
covers it. This notebook builds that retrieval layer and **measures** it, so the
choice of index, embedding model and query path rests on numbers from this
corpus rather than on what is fashionable.

The playbooks are not free prose. Each one is a structured record with a fixed
set of fields, and that structure decides the whole design:

| Field | Role |
| --- | --- |
| `alert_name` (+ aliases) | Routing key, and embedded. Confluence keeps the use case name and the alert name in one column, so this is both. |
| `rule_description` | Embedded |
| `true_condition` | Embedded — what makes this a real detection |
| `false_condition` | Embedded — what makes it noise |
| `use_case_category` | Metadata, and used as embedding context |
| `version` | Metadata — the playbook's own version |
| `splunk_query` | Metadata — parameters, not raw SPL (see section 1) |
| `work_around` | Metadata — the remediation steps (`Step to work` is the same field under another label) |
| `target_audience`, `contact` | Metadata — who owns it |

The split is the point: **embed what identifies which playbook applies, store
what the analyst then acts on.** Embedding `splunk_query` would fill the vector
space with `index=` and `stats count by`, tokens every playbook shares; embedding
`contact` would match on people's names; embedding `version` would add a number
with no semantic content at all.

`use_case_category` is the one that sits between the two. It is far too
low-cardinality to be its own record — a handful of categories across hundreds of
playbooks means those vectors would be near-identical and match everything
equally. But it is real context, so it goes into the *prefix* of the other
records instead, where it sharpens them without competing with them.

What gets measured, on a labelled query set:

| Metric | Why it matters here |
| --- | --- |
| `recall@k` | Did the right playbook reach the context at all? Nothing downstream fixes a miss. |
| `MRR` | How close to the top. A 7b model reads the first chunks most carefully. |
| query latency | The agent loop's budget is dominated by LLM turns — retrieval must stay in the noise. |
| tier distribution | Which query path actually serves production traffic. |

In [ ]:
from __future__ import annotations

import difflib
import json
import re
import statistics
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

# Optional pieces — the notebook reports what is missing instead of failing on
# import, so the parts that do work still run.
AVAILABLE: dict[str, bool] = {}
for name in ["sentence_transformers", "qdrant_client", "httpx"]:
    try:
        __import__(name)
        AVAILABLE[name] = True
    except ImportError:
        AVAILABLE[name] = False

print(json.dumps(AVAILABLE, indent=2))

## 1. The playbook record

`EMBEDDED_FIELDS` and `METADATA_FIELDS` below are the contract between
Confluence and everything downstream. Adding a field to Confluence means adding
it to one of those two lists — nothing else in the notebook needs to know.

### Why `splunk_query` holds parameters, not SPL

The obvious design is to store the analyst's search verbatim and have the agent
run it. That hands anyone with edit access to the Confluence space the ability to
run arbitrary SPL as the service account — and SPL is not read-only: `| delete`,
`| collect`, `| outputlookup` and `| script` all write or destroy.

So the field holds the *parameters* of a search, and `soc-mcp-splunk` builds the
SPL from them. It maps directly onto the `search_events` tool that connector
already exposes, so nothing new is needed on that side:

```json
{"index": "wineventlog", "sourcetype": "WinEventLog:Security",
 "filters": {"EventCode": "4625"}, "earliest": "-24h"}
```

If a search genuinely cannot be expressed this way, the answer is a reviewed
allowlist of SPL keyed by hash — not accepting whatever the wiki currently says.

### Loading from Confluence

Playbooks with this shape are normally authored with a **Page Properties** macro,
which stores each field as a row in a table. The loader below reads those rows.
Without credentials it falls back to six samples that mirror how real runbooks
read, so the notebook runs end to end today and the benchmark has something to
score.

In [ ]:
# The schema. Order matters: it is the order fields are scored and displayed.
EMBEDDED_FIELDS = ["alert_name", "rule_description", "true_condition", "false_condition"]
METADATA_FIELDS = ["use_case_category", "version", "splunk_query", "work_around",
                   "target_audience", "contact"]


@dataclass
class Playbook:
    page_id: str
    # Confluence keeps the use case name and the alert name in one column, so
    # this is both: the routing key, and what an analyst calls the detection.
    alert_name: str
    use_case_category: str = ""
    rule_description: str = ""
    true_condition: str = ""
    false_condition: str = ""
    # Alert names for the same detection under other names — FortiSOAR and
    # Confluence do not always agree, and detections get renamed.
    aliases: list[str] = field(default_factory=list)
    # metadata, never embedded
    version: str = ""
    splunk_query: dict | None = None
    work_around: str = ""
    target_audience: str = ""
    contact: str = ""
    source: str = "sample"

    def field_text(self, name: str) -> str:
        return (getattr(self, name, "") or "").strip()

    @property
    def metadata(self) -> dict:
        return {name: getattr(self, name) for name in METADATA_FIELDS}


CONFLUENCE_BASE_URL = ""      # e.g. "https://your-org.atlassian.net/wiki"
CONFLUENCE_SPACE_KEY = ""     # e.g. "SOC"
CONFLUENCE_EMAIL = ""
CONFLUENCE_API_TOKEN = ""

# Confluence field label -> our field name. Extend as the space evolves; a label
# that is not here is ignored rather than silently embedded.
FIELD_LABELS = {
    "alert name": "alert_name",
    "alert": "alert_name",
    "use case": "alert_name",             # same column as the alert name
    "use case / alert name": "alert_name",
    "use case name": "alert_name",
    "use case category": "use_case_category",
    "category": "use_case_category",
    "aliases": "aliases",
    "rule description": "rule_description",
    "description": "rule_description",   # older pages used the shorter label
    "true condition": "true_condition",
    "false condition": "false_condition",
    "version": "version",
    "splunk query": "splunk_query",
    "work around": "work_around",
    "workaround": "work_around",
    "step to work": "work_around",     # same meaning as work around
    "target audience": "target_audience",
    "contact": "contact",
    "contact info": "contact",
}


def _strip_html(fragment: str) -> str:
    text = re.sub(r"<br\s*/?>", "\n", fragment)
    text = re.sub(r"</(p|li|tr)>", "\n", text)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"[ \t]+", " ", text).strip()


def parse_page_properties(storage_xhtml: str) -> dict[str, str]:
    """Read a Page Properties macro: each row is <th>label</th><td>value</td>."""
    rows = re.findall(
        r"<t[hd][^>]*>(.*?)</t[hd]>\s*<t[hd][^>]*>(.*?)</t[hd]>",
        storage_xhtml,
        flags=re.DOTALL,
    )
    parsed = {}
    for raw_label, raw_value in rows:
        label = _strip_html(raw_label).lower().rstrip(":").strip()
        if label in FIELD_LABELS:
            parsed[FIELD_LABELS[label]] = _strip_html(raw_value)
    return parsed


def load_from_confluence(limit: int = 500) -> list[Playbook]:
    """Pull one space's pages. Returns [] when credentials are unset."""
    if not (CONFLUENCE_BASE_URL and CONFLUENCE_SPACE_KEY and CONFLUENCE_API_TOKEN):
        return []

    import httpx

    playbooks, start = [], 0
    auth = (CONFLUENCE_EMAIL, CONFLUENCE_API_TOKEN)
    with httpx.Client(base_url=CONFLUENCE_BASE_URL.rstrip("/"), auth=auth, timeout=60) as client:
        while len(playbooks) < limit:
            resp = client.get(
                "/rest/api/content",
                params={
                    "spaceKey": CONFLUENCE_SPACE_KEY,
                    "expand": "body.storage,version",
                    "limit": 50,
                    "start": start,
                },
            )
            resp.raise_for_status()
            page = resp.json()
            for result in page.get("results", []):
                storage = result.get("body", {}).get("storage", {}).get("value", "")
                fields = parse_page_properties(storage)
                raw_aliases = fields.pop("aliases", "")
                query = fields.pop("splunk_query", "")
                try:
                    parsed_query = json.loads(query) if query else None
                except json.JSONDecodeError:
                    # Raw SPL in the field: keep it visible but do not let it
                    # reach the Splunk connector as an executable search.
                    parsed_query = {"unparsed": query}
                playbooks.append(
                    Playbook(
                        page_id=result["id"],
                        alert_name=fields.pop("alert_name", result["title"]),
                        aliases=[a.strip() for a in re.split(r"[,\n]", raw_aliases) if a.strip()],
                        splunk_query=parsed_query,
                        source="confluence",
                        **fields,
                    )
                )
            if not page.get("_links", {}).get("next"):
                break
            start += 50
    return playbooks


SAMPLE_PLAYBOOKS = [
    Playbook(
        page_id="PB-001",
        alert_name="Excessive Failed Logins",
        use_case_category="Authentication",
        aliases=["Brute Force Access Behavior Detected", "T1110"],
        rule_description=(
            "Repeated failed authentication attempts against a single account or across "
            "many accounts from one source. Maps to MITRE ATT&CK T1110 Brute Force."
        ),
        true_condition=(
            "A successful login follows the burst of failures, or the source is external "
            "and reaching a VPN or OWA endpoint, or more than 10 distinct accounts are "
            "targeted from one source address."
        ),
        false_condition=(
            "Expired passwords on service accounts, a phone mail client retrying old "
            "credentials after a reset, or an authenticated vulnerability scan running "
            "with the wrong credential set. Under 20 failures in an hour from a corporate "
            "subnet is almost always one of these."
        ),
        version="2.1",
        splunk_query={
            "index": "wineventlog",
            "sourcetype": "WinEventLog:Security",
            "filters": {"EventCode": "4625"},
            "earliest": "-24h",
        },
        work_around=(
            "Lock the source address at the perimeter, force a password reset on any "
            "account that authenticated successfully, and confirm MFA is enforced."
        ),
        target_audience="Tier 1",
        contact="identity-team@soc.local",
    ),
    Playbook(
        page_id="PB-002",
        alert_name="Suspicious PowerShell Encoded Command",
        use_case_category="Endpoint",
        aliases=["Obfuscated Script Execution", "T1059.001"],
        rule_description=(
            "PowerShell invoked with -EncodedCommand or heavy obfuscation. Maps to "
            "T1059.001. Common in both commodity malware and legitimate management "
            "tooling, which is what makes it noisy."
        ),
        true_condition=(
            "The decoded payload contains a download cradle, the parent process is an "
            "Office application such as winword.exe or outlook.exe, or the process opens "
            "a connection to a host outside the corporate range."
        ),
        false_condition=(
            "SCCM and Intune management scripts, software packaging tools, and internal "
            "automation that legitimately passes encoded arguments."
        ),
        version="1.4",
        splunk_query={
            "index": "sysmon",
            "filters": {"EventCode": "1", "process_name": "powershell.exe"},
            "earliest": "-24h",
        },
        work_around=(
            "Isolate the host, capture the decoded command for the incident record, and "
            "check whether the parent process chain appears on other endpoints."
        ),
        target_audience="Tier 2",
        contact="endpoint-team@soc.local",
    ),
    Playbook(
        page_id="PB-003",
        alert_name="Impossible Travel Detected",
        use_case_category="Authentication",
        aliases=["Anomalous Login Location", "T1078"],
        rule_description=(
            "Two successful authentications for one account from locations too far apart "
            "for the time between them. Maps to T1078 Valid Accounts. Sourced from Azure "
            "AD or Okta sign-in logs."
        ),
        true_condition=(
            "No VPN or corporate proxy explains the second location, MFA was bypassed or "
            "satisfied by a push approved seconds after a foreign sign-in, or mailbox "
            "rules and OAuth consent grants were created immediately after."
        ),
        false_condition=(
            "Corporate VPN egress in another region, cloud-hosted mail clients, and users "
            "travelling with an active session that still refreshes from the old location."
        ),
        version="3.0",
        splunk_query={
            "index": "azuread",
            "filters": {"category": "SignInLogs"},
            "earliest": "-7d",
        },
        work_around=(
            "Revoke active sessions, force re-authentication, and review any mailbox "
            "rules created in the window."
        ),
        target_audience="Tier 2",
        contact="identity-team@soc.local",
    ),
    Playbook(
        page_id="PB-004",
        alert_name="Endpoint Malware Detection",
        use_case_category="Endpoint",
        aliases=["EDR Threat Detected", "T1204"],
        rule_description=(
            "EDR or antivirus flagged a file or process on an endpoint. Maps to T1204 "
            "User Execution. Severity depends far more on whether execution was blocked "
            "than on the detection name."
        ),
        true_condition=(
            "Execution was not blocked, the same hash appears on more than one host, or "
            "the file arrived by email and other recipients received it too."
        ),
        false_condition=(
            "Security testing tools and penetration-testing binaries on authorised hosts, "
            "and developer builds of unsigned internal software."
        ),
        version="1.9",
        splunk_query={"index": "edr", "earliest": "-24h"},
        work_around=(
            "Quarantine the file, isolate the host if execution succeeded, and sweep the "
            "estate for the hash."
        ),
        target_audience="Tier 1",
        contact="endpoint-team@soc.local",
    ),
    Playbook(
        page_id="PB-005",
        alert_name="Large Outbound Data Transfer",
        use_case_category="Data Loss",
        aliases=["Potential Data Exfiltration", "T1048"],
        rule_description=(
            "Unusual volume of outbound data from one host or user. Maps to T1048 "
            "Exfiltration Over Alternative Protocol. Baseline matters more than the "
            "absolute number."
        ),
        true_condition=(
            "The destination is not a sanctioned service, the traffic rides DNS or ICMP "
            "rather than HTTPS, or the user is under HR review."
        ),
        false_condition=(
            "Backup jobs, cloud sync clients, large legitimate uploads to sanctioned "
            "storage, and software distribution to remote sites."
        ),
        version="2.0",
        splunk_query={"index": "firewall", "earliest": "-24h"},
        work_around=(
            "Block the destination, preserve the netflow for the incident record, and "
            "confirm with the data owner whether the transfer was authorised."
        ),
        target_audience="Tier 2",
        contact="netsec-team@soc.local",
    ),
    Playbook(
        page_id="PB-006",
        alert_name="User Reported Phishing",
        use_case_category="Email",
        aliases=["Suspicious Email Reported", "T1566"],
        rule_description=(
            "A user reported a suspicious email through the report button. Maps to T1566 "
            "Phishing. High value because a human already applied judgement."
        ),
        true_condition=(
            "Any recipient clicked through, credentials were submitted, or the campaign "
            "reached more than 10 mailboxes."
        ),
        false_condition=(
            "Marketing email, internal phishing simulation exercises, and legitimate mail "
            "from a partner whose SPF record is misconfigured."
        ),
        version="2.3",
        splunk_query={"index": "email", "earliest": "-7d"},
        work_around=(
            "Purge the campaign from all mailboxes, block the sender and URL, and reset "
            "credentials for anyone who submitted them."
        ),
        target_audience="Tier 1",
        contact="email-security@soc.local",
    ),
]

playbooks = load_from_confluence() or SAMPLE_PLAYBOOKS
print(f"{len(playbooks)} playbooks, source={playbooks[0].source}\n")
for pb in playbooks[:8]:
    print(f"  {pb.page_id}  v{pb.version:4s}  [{pb.use_case_category}]  {pb.alert_name}")
    print(f"           aliases: {', '.join(pb.aliases) or '-'}")

categories: dict[str, int] = {}
for pb in playbooks:
    categories[pb.use_case_category or "(uncategorised)"] = (
        categories.get(pb.use_case_category or "(uncategorised)", 0) + 1
    )
print(f"\nplaybooks per category: {categories}")

## 2. One record per field

A playbook does not become one vector. Each embedded field becomes its own
record, so a query lands on the field that answers it.

The reason is visible in the schema. "Is this a false positive when the source is
internal?" should score against `false_condition` directly. Concatenating the
four fields into a single vector averages that signal away — `description` is the
longest field and would dominate every comparison.

Two things follow, both of them useful:

- Scores are **max-pooled back to the playbook**, so the answer is still one
  playbook per query, not four fragments of one.
- The winning field is known, so the agent can be told *why* a playbook matched —
  "this matches the false-positive criteria" is a different instruction from
  "this matches the description".

Each record is prefixed with the alert name and the field it came from, for the
same reason: an embedding of the words "the source is external" alone says
nothing about which detection it belongs to.

In [ ]:
FIELD_LABEL = {
    "alert_name": "alert",
    "rule_description": "rule description",
    "true_condition": "true positive criteria",
    "false_condition": "false positive criteria",
}


@dataclass
class Record:
    record_id: str
    page_id: str
    alert_name: str
    use_case_category: str
    field_name: str
    text: str

    @property
    def embed_text(self) -> str:
        """Identity first, then the field's content."""
        head = f"{self.use_case_category} / {self.alert_name}" if self.use_case_category else self.alert_name
        return f"{head} — {FIELD_LABEL[self.field_name]}\n{self.text}"


def build_records(playbooks: list[Playbook]) -> list[Record]:
    records = []
    for pb in playbooks:
        for name in EMBEDDED_FIELDS:
            text = pb.field_text(name)
            if not text:
                continue
            if name == "alert_name":
                # Aliases belong with the name they alias, not as separate records.
                text = " / ".join([text, *pb.aliases])
            records.append(
                Record(
                    record_id=f"{pb.page_id}#{name}",
                    page_id=pb.page_id,
                    alert_name=pb.alert_name,
                    use_case_category=pb.use_case_category,
                    field_name=name,
                    text=text,
                )
            )
    return records


records = build_records(playbooks)
by_page = {pb.page_id: pb for pb in playbooks}

per_field: dict[str, int] = {}
for record in records:
    per_field[record.field_name] = per_field.get(record.field_name, 0) + 1

lengths = [len(r.text) for r in records]
print(f"{len(records)} records from {len(playbooks)} playbooks")
print(f"per field: {per_field}")
print(f"chars: min={min(lengths)} median={int(statistics.median(lengths))} max={max(lengths)}")
print()
print("example record:")
print(records[3].embed_text[:320])

## 3. Embeddings

Two backends, chosen by what is installed:

- **sentence-transformers**, in-process. `bge-small-en-v1.5` is 384-dimensional
  and built for retrieval; `bge-base` is 768-dimensional and slower. Both get
  scored below rather than assumed.
- **Ollama**, over HTTP to the `soc-ollama` container already in the stack —
  slower per call, but it is the path production takes.

BGE models expect an instruction prefix on the *query* and none on the document.
Skipping it is a quiet 2-5 point recall loss, so it is applied here.

Vectors are L2-normalised at encode time, which turns cosine similarity into a
plain dot product and makes search one matrix multiply.

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
OLLAMA_URL = "http://localhost:11434"
OLLAMA_EMBED_MODEL = "nomic-embed-text"


class Embedder:
    """One interface over sentence-transformers and Ollama."""

    def __init__(self, name: str, backend: str = "st", query_prefix: str = ""):
        self.name = name
        self.backend = backend
        self.query_prefix = query_prefix
        self._model = None

    def _load(self):
        if self._model is None and self.backend == "st":
            from sentence_transformers import SentenceTransformer

            self._model = SentenceTransformer(self.name)
        return self._model

    def encode(self, texts: list[str], is_query: bool = False) -> np.ndarray:
        if is_query and self.query_prefix:
            texts = [self.query_prefix + t for t in texts]

        if self.backend == "st":
            vectors = self._load().encode(
                texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False
            )
            return np.asarray(vectors, dtype=np.float32)

        import httpx

        vectors = []
        with httpx.Client(base_url=OLLAMA_URL, timeout=120) as client:
            for text in texts:
                resp = client.post("/api/embed", json={"model": self.name, "input": text})
                resp.raise_for_status()
                vectors.append(resp.json()["embeddings"][0])
        matrix = np.asarray(vectors, dtype=np.float32)
        return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


CANDIDATES = []
if AVAILABLE["sentence_transformers"]:
    CANDIDATES += [
        Embedder("BAAI/bge-small-en-v1.5", "st", QUERY_PREFIX),
        Embedder("BAAI/bge-base-en-v1.5", "st", QUERY_PREFIX),
    ]
if AVAILABLE["httpx"]:
    CANDIDATES.append(Embedder(OLLAMA_EMBED_MODEL, "ollama"))

corpus_texts = [r.embed_text for r in records]
embeddings: dict[str, np.ndarray] = {}

for embedder in CANDIDATES:
    try:
        start = time.perf_counter()
        matrix = embedder.encode(corpus_texts)
        elapsed = time.perf_counter() - start
    except Exception as exc:  # model download blocked, Ollama down, etc.
        print(f"skip {embedder.name}: {type(exc).__name__}: {str(exc)[:100]}")
        continue

    embeddings[embedder.name] = matrix
    print(f"{embedder.name:32s} dim={matrix.shape[1]:4d}  {elapsed:.2f}s for {len(records)} records")

WORKING = [e for e in CANDIDATES if e.name in embeddings]
assert WORKING, "no embedding backend available — install sentence-transformers or start Ollama"

## 4. Search: exact vs approximate, dense vs lexical

**Exact beats approximate at this scale.** Search over `n` normalised vectors is
one `(n, d) @ (d,)` product — a few hundred microseconds of BLAS for a corpus
this size, and exact by construction, with no graph to build or rebuild on every
Confluence sync. HNSW trades recall for sublinear scaling, which starts paying
around `n ≈ 10⁵`. Both are timed, along with a synthetic corpus large enough to
show where the crossover actually sits.

**Lexical still matters.** Dense embeddings generalise, which is the failure mode
on SOC text: `T1110`, `EventCode=4625` and `New-InboxRule` are exact tokens, and
an embedding places `T1110` near `T1078` because both are technique identifiers
in similar sentences — precisely wrong when the alert names one of them. BM25
scores exact overlap, weighting rare terms heavily, so it is strongest where
embeddings are weakest.

They are combined with **Reciprocal Rank Fusion**: the two score scales are not
comparable (a dot product in [-1, 1] against an unbounded BM25 sum), and
normalising depends on the spread of whichever query was asked. RRF throws the
scores away and keeps the ranks, `1 / (k + rank)` each. `k = 60` is from the
original paper and is not sensitive.

In [ ]:
K1, B = 1.5, 0.75   # BM25 parameters
RRF_K = 60


class FlatIndex:
    """Exact cosine search. Normalised vectors, so a dot product is the score."""

    def __init__(self, matrix: np.ndarray):
        self.matrix = np.ascontiguousarray(matrix)

    def search(self, query: np.ndarray, k: int) -> list[tuple[int, float]]:
        scores = self.matrix @ query
        # argpartition beats a full sort: only the top k need ordering.
        top = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
        top = top[np.argsort(-scores[top])]
        return [(int(i), float(scores[i])) for i in top]


def tokenize(text: str) -> list[str]:
    """Keep the shapes that matter in SOC text: T1110, EventCode=4625, file.exe."""
    return re.findall(r"[a-z0-9][a-z0-9._\-]*", text.lower())


class BM25Index:
    def __init__(self, documents: list[str]):
        self.docs = [tokenize(d) for d in documents]
        self.doc_lens = np.array([len(d) for d in self.docs], dtype=np.float32)
        self.avg_len = float(self.doc_lens.mean())

        vocabulary: dict[str, int] = {}
        for doc in self.docs:
            for term in set(doc):
                vocabulary.setdefault(term, len(vocabulary))
        self.vocabulary = vocabulary

        self.tf = np.zeros((len(self.docs), len(vocabulary)), dtype=np.float32)
        for row, doc in enumerate(self.docs):
            for term in doc:
                self.tf[row, vocabulary[term]] += 1

        doc_freq = (self.tf > 0).sum(axis=0)
        n = len(self.docs)
        self.idf = np.log(1 + (n - doc_freq + 0.5) / (doc_freq + 0.5)).astype(np.float32)
        # Length normalisation is query-independent — precompute it.
        self.denominator = K1 * (1 - B + B * self.doc_lens / self.avg_len)

    def search(self, query: str, k: int) -> list[tuple[int, float]]:
        columns = [self.vocabulary[t] for t in tokenize(query) if t in self.vocabulary]
        if not columns:
            return []
        tf = self.tf[:, columns]
        scores = (
            self.idf[columns] * (tf * (K1 + 1)) / (tf + self.denominator[:, None])
        ).sum(axis=1)
        top = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
        top = top[np.argsort(-scores[top])]
        return [(int(i), float(scores[i])) for i in top if scores[i] > 0]


def rrf_fuse(rankings: list[list[tuple[int, float]]], k: int) -> list[tuple[int, float]]:
    fused: dict[int, float] = {}
    for ranking in rankings:
        for rank, (idx, _score) in enumerate(ranking):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)
    return sorted(fused.items(), key=lambda kv: -kv[1])[:k]


def time_search(index, queries: np.ndarray, k: int = 5, repeats: int = 20) -> dict:
    timings = []
    for _ in range(repeats):
        for query in queries:
            start = time.perf_counter()
            index.search(query, k)
            timings.append((time.perf_counter() - start) * 1000)
    timings.sort()
    return {
        "p50_ms": round(timings[len(timings) // 2], 3),
        "p95_ms": round(timings[int(len(timings) * 0.95)], 3),
    }


bm25 = BM25Index(corpus_texts)
primary = WORKING[0]
matrix = embeddings[primary.name]
probe = primary.encode(["brute force failed logins from an external IP"], is_query=True)

print(f"corpus: {matrix.shape[0]} records x {matrix.shape[1]} dims")
print(f"bm25 vocabulary: {len(bm25.vocabulary)} terms\n")
print("flat (exact)   ", time_search(FlatIndex(matrix), probe))

if AVAILABLE["qdrant_client"]:
    from qdrant_client import QdrantClient, models

    build_start = time.perf_counter()
    client = QdrantClient(":memory:")
    client.create_collection(
        "records",
        vectors_config=models.VectorParams(size=matrix.shape[1], distance=models.Distance.COSINE),
    )
    client.upload_collection("records", vectors=matrix, ids=list(range(len(matrix))))
    build_s = time.perf_counter() - build_start

    class QdrantIndex:
        def search(self, query, k):
            hits = client.query_points("records", query=query, limit=k).points
            return [(int(h.id), float(h.score)) for h in hits]

    print("qdrant (HNSW)  ", time_search(QdrantIndex(), probe), f"build={build_s:.2f}s")

print("\nflat search as the corpus grows:")
rng = np.random.default_rng(RANDOM_SEED)
for n in [len(records), 1_000, 10_000, 100_000]:
    synthetic = rng.normal(size=(n, matrix.shape[1])).astype(np.float32)
    synthetic /= np.linalg.norm(synthetic, axis=1, keepdims=True)
    stats = time_search(FlatIndex(synthetic), probe, repeats=5)
    print(f"  n={n:7d}  p50={stats['p50_ms']:7.3f} ms  p95={stats['p95_ms']:7.3f} ms")

In [ ]:
@dataclass
class Hit:
    playbook: Playbook
    score: float
    matched_field: str
    matched_text: str


class Retriever:
    """Field-level search, max-pooled back to whole playbooks."""

    def __init__(self, embedder: Embedder, dense: FlatIndex, lexical: BM25Index,
                 records: list[Record], by_page: dict[str, Playbook]):
        self.embedder = embedder
        self.dense = dense
        self.lexical = lexical
        self.records = records
        self.by_page = by_page

    def search(self, query: str, k: int = 3, mode: str = "hybrid") -> list[Hit]:
        pool = max(k * 8, 24)
        dense_hits = lexical_hits = []

        if mode in ("dense", "hybrid"):
            vector = self.embedder.encode([query], is_query=True)[0]
            dense_hits = self.dense.search(vector, pool)
        if mode in ("lexical", "hybrid"):
            lexical_hits = self.lexical.search(query, pool)

        if mode == "dense":
            ranked = dense_hits
        elif mode == "lexical":
            ranked = lexical_hits
        else:
            ranked = rrf_fuse([dense_hits, lexical_hits], pool)

        # Max-pool: a playbook scores as its best-matching field.
        best: dict[str, tuple[float, Record]] = {}
        for index, score in ranked:
            record = self.records[index]
            if record.page_id not in best or score > best[record.page_id][0]:
                best[record.page_id] = (score, record)

        ordered = sorted(best.items(), key=lambda kv: -kv[1][0])[:k]
        return [
            Hit(self.by_page[page_id], score, record.field_name, record.text)
            for page_id, (score, record) in ordered
        ]


retriever = Retriever(primary, FlatIndex(matrix), bm25, records, by_page)

for query in [
    "is it a false positive if the scanner is authenticated",
    "T1059.001",
    "account signed in from two countries minutes apart",
]:
    print(f"{query!r}")
    for hit in retriever.search(query, k=2):
        print(f"   {hit.playbook.page_id}  {hit.matched_field:16s} {hit.playbook.alert_name}")
    print()

## 5. Routing

Everything above treats this as open-domain search. It is not. FortiSOAR does not
send questions — it sends the name of a correlation search, from a set that is
finite, enumerable, and changes only when someone edits a detection.

For a name already known, retrieval is a dictionary lookup: no embedding, no
search, exact by construction. That removes the query-embedding cost that
section 4 shows to dominate dense latency, for what should be most traffic.

The table is **built from the corpus, never written by hand**: `alert_name` plus
`aliases`, straight out of Confluence. Names are normalised before matching, and
a near miss falls through to fuzzy matching — a renamed detection should not
silently lose its playbook, and the fuzzy hit is reported so the page can be
fixed.

In [ ]:
FUZZY_CUTOFF = 0.85


def normalize_alert(name: str) -> str:
    """Case and spacing folded; dots and hyphens kept so T1059.001 survives."""
    cleaned = re.sub(r"[^a-z0-9.\- ]+", " ", name.lower())
    return re.sub(r"\s+", " ", cleaned).strip()


class AlertRouter:
    """alert name -> playbook, rebuilt from the corpus on every sync."""

    def __init__(self, playbooks: list[Playbook]):
        self.table: dict[str, tuple[str, str, str]] = {}   # key -> (page_id, display, source)
        self.conflicts: list[tuple[str, str, str]] = []

        for pb in playbooks:
            for name, source in [(pb.alert_name, "alert_name"),
                                 *[(a, "alias") for a in pb.aliases]]:
                key = normalize_alert(name)
                if not key:
                    continue
                existing = self.table.get(key)
                if existing and existing[0] != pb.page_id:
                    # A primary name outranks an alias; equal ranks are a real
                    # ambiguity and get reported rather than silently resolved.
                    if source == "alert_name" and existing[2] == "alias":
                        self.table[key] = (pb.page_id, name, source)
                    elif existing[2] == "alert_name" and source == "alias":
                        pass
                    else:
                        self.conflicts.append((name, existing[0], pb.page_id))
                    continue
                self.table[key] = (pb.page_id, name, source)

        self._keys = list(self.table)

    def lookup(self, alert_name: str) -> tuple[str | None, str]:
        key = normalize_alert(alert_name)
        if key in self.table:
            return self.table[key][0], "exact"
        near = difflib.get_close_matches(key, self._keys, n=1, cutoff=FUZZY_CUTOFF)
        if near:
            return self.table[near[0]][0], "fuzzy"
        return None, "miss"


router = AlertRouter(playbooks)

by_source: dict[str, int] = {}
for _page, _display, source in router.table.values():
    by_source[source] = by_source.get(source, 0) + 1

print(f"{len(router.table)} alert names mapped to {len(playbooks)} playbooks")
print(f"by source: {by_source}")
print("conflicts:", router.conflicts or "none")
print()
for probe_name in ["Excessive Failed Logins", "excessive  failed logins", "T1059.001",
                   "Excessive Failed Login Attempts", "Cloud Storage Bucket Made Public"]:
    page_id, how = router.lookup(probe_name)
    print(f"  {probe_name:36s} -> {page_id or '-':7s} ({how})")

### Coverage: which alerts have no playbook

Point the router at the alert names that have actually arrived and the gaps fall
out — every detection firing into the SOC that nobody has written a runbook for.
That list is worth more to the SOC than any retrieval metric on this page.

The core backend already records them. Export the real ones with:

```sql
SELECT alert_name, COUNT(*) AS n
FROM ai_jobs
GROUP BY alert_name
ORDER BY n DESC;
```

Save it as `observed_alerts.json` (`[{"alert_name": ..., "count": ...}]`) next to
this notebook. That query touches only the name and a count — no `raw_log`, so
no security log content leaves the server.

In [ ]:
OBSERVED_ALERTS_PATH = Path("observed_alerts.json")

SAMPLE_OBSERVED = [
    {"alert_name": "Excessive Failed Logins", "count": 412},
    {"alert_name": "excessive failed logins", "count": 38},           # casing drift
    {"alert_name": "Excessive Failed Login Attempts", "count": 27},   # renamed detection
    {"alert_name": "Suspicious PowerShell Encoded Command", "count": 190},
    {"alert_name": "Obfuscated Script Execution", "count": 64},
    {"alert_name": "Impossible Travel Detected", "count": 143},
    {"alert_name": "Endpoint Malware Detection", "count": 221},
    {"alert_name": "EDR Threat Detected", "count": 88},
    {"alert_name": "Large Outbound Data Transfer", "count": 76},
    {"alert_name": "User Reported Phishing", "count": 305},
    {"alert_name": "Cloud Storage Bucket Made Public", "count": 52},  # no playbook
    {"alert_name": "Kerberoasting Activity Detected", "count": 31},   # no playbook
]

if OBSERVED_ALERTS_PATH.exists():
    observed = json.loads(OBSERVED_ALERTS_PATH.read_text(encoding="utf-8"))
    print(f"loaded {len(observed)} observed alert names from {OBSERVED_ALERTS_PATH}")
else:
    observed = SAMPLE_OBSERVED
    print(f"{OBSERVED_ALERTS_PATH} not found — using {len(observed)} sample alert names")

buckets: dict[str, list] = {"exact": [], "fuzzy": [], "miss": []}
for entry in observed:
    page_id, how = router.lookup(entry["alert_name"])
    buckets[how].append((entry["alert_name"], entry["count"], page_id))

total_volume = sum(e["count"] for e in observed)
print()
for how in ["exact", "fuzzy", "miss"]:
    volume = sum(count for _n, count, _p in buckets[how])
    share = volume / total_volume * 100 if total_volume else 0
    print(f"{how:6s} {len(buckets[how]):3d} names  {volume:6d} alerts  {share:5.1f}% of volume")

if buckets["fuzzy"]:
    print("\nfuzzy — a detection was renamed and Confluence was not updated:")
    for name, count, page_id in buckets["fuzzy"]:
        print(f"  {name!r} -> {page_id}  ({count} alerts)")

if buckets["miss"]:
    print("\nNO PLAYBOOK — ranked by how often they fire:")
    for name, count, _p in sorted(buckets["miss"], key=lambda x: -x[1]):
        print(f"  {count:5d}  {name}")

## 6. Tiered retrieval

Three tiers, cheapest first, each falling through only when it cannot answer:

| Tier | Cost | Handles |
| --- | --- | --- |
| 1. Router, exact | ~0 ms | A known alert name |
| 2. Router, fuzzy | ~0 ms | A renamed detection |
| 3. Hybrid | ~25 ms | Genuinely novel wording, or a question rather than a name |

In front of tier 3 sits an embedding cache keyed on the normalised query, for the
same reason the router exists: alert text repeats, and a cache hit skips the
forward pass that is the entire cost of that tier.

The comparison is against always running the hybrid retriever.

In [ ]:
class CachingEmbedder:
    """Wraps an embedder with a dict on the normalised query."""

    def __init__(self, embedder: Embedder):
        self.embedder = embedder
        self.name, self.backend = embedder.name, embedder.backend
        self.query_prefix = embedder.query_prefix
        self.cache: dict[str, np.ndarray] = {}
        self.hits = self.misses = 0

    def encode(self, texts: list[str], is_query: bool = False) -> np.ndarray:
        if not is_query:
            return self.embedder.encode(texts, is_query=False)
        out = []
        for text in texts:
            key = normalize_alert(text)
            if key in self.cache:
                self.hits += 1
            else:
                self.misses += 1
                self.cache[key] = self.embedder.encode([text], is_query=True)[0]
            out.append(self.cache[key])
        return np.vstack(out)

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total else 0.0


class TieredRetriever:
    def __init__(self, router: AlertRouter, hybrid: Retriever, by_page: dict[str, Playbook]):
        self.router = router
        self.hybrid = hybrid
        self.by_page = by_page

    def search(self, alert_name: str, question: str = "", k: int = 3):
        page_id, how = self.router.lookup(alert_name)
        if page_id and not question:
            pb = self.by_page[page_id]
            return [Hit(pb, 1.0, "alert_name", pb.alert_name)], f"router/{how}"
        query = f"{alert_name} {question}".strip()
        return self.hybrid.search(query, k=k, mode="hybrid"), "hybrid"


best_embedder = WORKING[0]
cached = CachingEmbedder(best_embedder)
hybrid_cached = Retriever(
    cached, FlatIndex(embeddings[best_embedder.name]), bm25, records, by_page
)
tiered = TieredRetriever(router, hybrid_cached, by_page)
baseline = Retriever(
    best_embedder, FlatIndex(embeddings[best_embedder.name]), bm25, records, by_page
)

# Replay observed alerts in proportion to how often each actually fires.
workload = [e["alert_name"] for e in observed for _ in range(max(1, e["count"] // 10))]
print(f"workload: {len(workload)} alerts, {len(set(workload))} distinct names\n")

tier_counts: dict[str, int] = {}
tiered_ms, baseline_ms = [], []
for alert_name in workload:
    start = time.perf_counter()
    _hits, tier = tiered.search(alert_name)
    tiered_ms.append((time.perf_counter() - start) * 1000)
    tier_counts[tier] = tier_counts.get(tier, 0) + 1

    start = time.perf_counter()
    baseline.search(alert_name, k=3, mode="hybrid")
    baseline_ms.append((time.perf_counter() - start) * 1000)


def summarize(timings: list[float]) -> dict:
    ordered = sorted(timings)
    return {
        "mean_ms": round(sum(ordered) / len(ordered), 2),
        "p50_ms": round(ordered[len(ordered) // 2], 3),
        "p95_ms": round(ordered[int(len(ordered) * 0.95)], 2),
    }


print("tier distribution:")
for tier, count in sorted(tier_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {tier:15s} {count:5d}  {count / len(workload) * 100:5.1f}%")

tiered_stats, baseline_stats = summarize(tiered_ms), summarize(baseline_ms)
print(f"\n{'':10s} {'mean':>9s} {'p50':>9s} {'p95':>9s}")
print(f"{'tiered':10s} {tiered_stats['mean_ms']:8.2f}ms {tiered_stats['p50_ms']:8.3f}ms "
      f"{tiered_stats['p95_ms']:8.2f}ms")
print(f"{'hybrid':10s} {baseline_stats['mean_ms']:8.2f}ms {baseline_stats['p50_ms']:8.3f}ms "
      f"{baseline_stats['p95_ms']:8.2f}ms")
print(f"\nmean speedup: {baseline_stats['mean_ms'] / tiered_stats['mean_ms']:.0f}x")
print(f"embedding cache: {cached.hits}/{cached.hits + cached.misses} = "
      f"{cached.hit_rate * 100:.1f}% hit rate")

## 7. Benchmark

The labelled set covers the shapes a real alert produces, plus the shape that
field-level embedding exists to serve — a question aimed at one specific field.

`recall@k` counts a query as hit when the gold playbook appears in the top k;
`MRR` is the reciprocal rank of the first correct one. The `field` column records
which field won, which is how the field split gets validated rather than assumed.

With six playbooks these numbers are indicative, not conclusive. Re-run on the
real Confluence export before deciding anything.

In [ ]:
# (query, gold page_id, the field expected to win)
EVAL_SET: list[tuple[str, str, str]] = [
    # exact names, as FortiSOAR sends them
    ("Excessive Failed Logins", "PB-001", "alert_name"),
    ("User Reported Phishing", "PB-006", "alert_name"),
    ("T1110", "PB-001", "alert_name"),
    ("T1059.001", "PB-002", "alert_name"),
    # paraphrases with no shared vocabulary
    ("someone is guessing passwords against our VPN", "PB-001", "rule_description"),
    ("account signed in from two countries minutes apart", "PB-003", "rule_description"),
    ("a workstation is uploading gigabytes to an unknown server", "PB-005", "rule_description"),
    ("antivirus caught a file on a laptop", "PB-004", "rule_description"),
    # aimed at true-positive criteria
    ("the decoded payload downloads a second stage", "PB-002", "true_condition"),
    ("a login succeeded right after the failures", "PB-001", "true_condition"),
    ("the same hash is on several machines", "PB-004", "true_condition"),
    ("credentials were submitted by a recipient", "PB-006", "true_condition"),
    # aimed at false-positive criteria
    ("could this just be an authenticated vulnerability scan", "PB-001", "false_condition"),
    ("is SCCM known to run encoded commands", "PB-002", "false_condition"),
    ("the user is travelling with an active session", "PB-003", "false_condition"),
    ("this might be a phishing simulation exercise", "PB-006", "false_condition"),
    ("backup jobs move a lot of data too", "PB-005", "false_condition"),
]

K = 3


def evaluate(retriever: Retriever, mode: str, k: int = K) -> dict:
    hits, reciprocal, latencies, field_correct = 0, [], [], 0

    for query, gold, expected_field in EVAL_SET:
        start = time.perf_counter()
        results = retriever.search(query, k=k, mode=mode)
        latencies.append((time.perf_counter() - start) * 1000)

        ranks = [i for i, h in enumerate(results) if h.playbook.page_id == gold]
        if ranks:
            hits += 1
            reciprocal.append(1 / (ranks[0] + 1))
            if results[ranks[0]].matched_field == expected_field:
                field_correct += 1
        else:
            reciprocal.append(0.0)

    latencies.sort()
    return {
        f"recall@{k}": round(hits / len(EVAL_SET), 3),
        "mrr": round(sum(reciprocal) / len(EVAL_SET), 3),
        "field": round(field_correct / len(EVAL_SET), 3),
        "p50_ms": round(latencies[len(latencies) // 2], 1),
    }


rows = []
for embedder in WORKING:
    index = FlatIndex(embeddings[embedder.name])
    candidate = Retriever(embedder, index, bm25, records, by_page)
    for mode in ["dense", "lexical", "hybrid"]:
        if mode == "lexical" and any(r["mode"] == "lexical" for r in rows):
            continue   # BM25 does not depend on the embedding model
        rows.append({"embedder": embedder.name, "mode": mode, **evaluate(candidate, mode)})

header = f"{'embedder':32s} {'mode':8s} {'recall@3':>9s} {'mrr':>6s} {'field':>6s} {'p50 ms':>8s}"
print(header)
print("-" * len(header))
for row in rows:
    name = row["embedder"] if row["mode"] != "lexical" else "-"
    print(f"{name:32s} {row['mode']:8s} {row[f'recall@{K}']:9.3f} {row['mrr']:6.3f} "
          f"{row['field']:6.3f} {row['p50_ms']:8.1f}")

best = max(rows, key=lambda r: (r[f"recall@{K}"], r["mrr"], r["field"]))
print(f"\nbest: {best['embedder']} / {best['mode']}")

In [ ]:
# Where it still goes wrong, and which field answered when it went right.
# This list is the useful output of the whole notebook — it says what to fix.
best_embedder = next(e for e in WORKING if e.name == best["embedder"])
checker = Retriever(best_embedder, FlatIndex(embeddings[best_embedder.name]), bm25,
                    records, by_page)

print("wrong playbook:")
misses = 0
for query, gold, _expected in EVAL_SET:
    results = checker.search(query, k=K, mode=best["mode"])
    if not any(h.playbook.page_id == gold for h in results):
        misses += 1
        print(f"  {query!r}\n    want {gold}, got {', '.join(h.playbook.page_id for h in results)}")
print("  none" if not misses else "")

print("\nright playbook, unexpected field — check the field split, not the retrieval:")
surprises = 0
for query, gold, expected in EVAL_SET:
    results = checker.search(query, k=K, mode=best["mode"])
    matched = next((h for h in results if h.playbook.page_id == gold), None)
    if matched and matched.matched_field != expected:
        surprises += 1
        print(f"  {query!r}\n    expected {expected}, matched {matched.matched_field}")
print("  none" if not surprises else "")

## 8. Export

Four artifacts, all rebuilt by re-running this notebook — nothing downstream is
hand-maintained:

| File | Rebuilt when |
| --- | --- |
| `records.jsonl` | A Confluence field changes |
| `embeddings.npy` | Same, or the embedding model changes |
| `playbooks.json` | Any page changes — carries the metadata fields |
| `alert_map.json` | An alert name or alias changes |
| `manifest.json` | Every build — it is what the service validates against |

The manifest matters more than it looks. A vector matrix is meaningless without
the model that produced it, and a silent embedding-model change degrades
retrieval without ever raising an error: queries and documents simply stop living
in the same space. The service that loads these should refuse to start when the
manifest does not match the model it is configured with.

In [ ]:
export_matrix = embeddings[best_embedder.name]
np.save(ARTIFACT_DIR / "embeddings.npy", export_matrix)

with open(ARTIFACT_DIR / "records.jsonl", "w", encoding="utf-8") as fh:
    for record in records:
        fh.write(json.dumps(record.__dict__, ensure_ascii=False) + "\n")

(ARTIFACT_DIR / "playbooks.json").write_text(
    json.dumps([pb.__dict__ for pb in playbooks], indent=2, ensure_ascii=False),
    encoding="utf-8",
)

alert_map = {
    key: {"page_id": page_id, "display_name": display, "source": source}
    for key, (page_id, display, source) in router.table.items()
}
(ARTIFACT_DIR / "alert_map.json").write_text(
    json.dumps(alert_map, indent=2, ensure_ascii=False), encoding="utf-8"
)

manifest = {
    "embedding_model": best_embedder.name,
    "backend": best_embedder.backend,
    "query_prefix": best_embedder.query_prefix,
    "dimensions": int(export_matrix.shape[1]),
    "embedded_fields": EMBEDDED_FIELDS,
    "metadata_fields": METADATA_FIELDS,
    "record_count": len(records),
    "playbook_count": len(playbooks),
    "source": playbooks[0].source,
    "retrieval_mode": best["mode"],
    "rrf_k": RRF_K,
    "bm25": {"k1": K1, "b": B},
    "routing": {
        "alert_names": len(router.table),
        "by_source": by_source,
        "fuzzy_cutoff": FUZZY_CUTOFF,
        "conflicts": len(router.conflicts),
    },
    "eval": {k: v for k, v in best.items() if k not in ("embedder", "mode")},
    "built_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
(ARTIFACT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))
print(f"\nwritten to {ARTIFACT_DIR.resolve()}")

## What this feeds

Two containers, split because their credentials and failure modes differ — not
because of scale:

```
soc-confluence-sync   holds the Confluence token, writes artifacts, runs on a schedule
        │  atomic publish (write to a temp dir, then rename)
        ▼
   /artifacts (volume)
        │  loaded into memory at start, reloaded when the manifest changes
        ▼
soc-mcp-confluence    holds no Confluence credentials, serves find_playbook over MCP
```

A Confluence token has no business living in a container that accepts requests.
That, rather than throughput, is what the split buys.

### Still to settle

- **Re-run on the real export.** Six playbooks cannot separate a 384-dimensional
  model from a 768-dimensional one; a few hundred real pages will.
- **Incremental sync.** Confluence returns a version number per page — store a
  content hash per record and re-embed only what moved, once a full rebuild
  stops being seconds.
- **Coverage gaps are a work queue.** Section 5's miss list is the set of pages
  someone should write. Re-running after each sync turns it into a standing
  report.
- **Recurring fuzzy hits mean a page is stale.** Correct behaviour at runtime,
  wrong state to stay in — fix the alias in Confluence.

### One caution to carry into the connector

A Confluence page is editable by anyone with write access to the space, and
retrieved text lands directly in the model's context. Treat it as data, never as
instructions: the connector should return playbook content clearly delimited as
reference material, and the agent's system prompt should state that retrieved
pages never carry instructions. The same reasoning is why `splunk_query` holds
parameters rather than SPL.